In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import matplotlib.pyplot as plt
import os

BASE_PATH = "/content/drive/MyDrive/부산_BigData_공모전/"

final = pd.read_csv(
    BASE_PATH + "03_analysis_output/최종_유휴공유재산_우선검토대상_186건.csv",
    encoding="utf-8-sig"
)

print("shape:", final.shape)

print("\n[추천유형]")
print(final["대표추천유형"].value_counts())

print("\n[구군별 후보]")
print(final["gugun"].value_counts())

print("\n[행정동별 후보 상위]")
print(final["행정동명"].value_counts().head(20))

print("\n좌표 결측")
print(final[["x", "y"]].isna().sum())

In [ ]:
# Drive 안의 공간경계 관련 파일 탐색
target_ext = (".geojson", ".json", ".shp")

geo_files = []

for root, dirs, files in os.walk(BASE_PATH):
    for file in files:
        if file.lower().endswith(target_ext):
            geo_files.append(
                os.path.join(root, file)
            )

print("찾은 공간파일 수:", len(geo_files))

for path in geo_files[:50]:
    print(path)

In [ ]:
# 현재 admdongkor 버전의 컬럼 구조를 확인하고 부산 경계 자동 추출

print("[전체 컬럼]")
print(boundary_all.columns.tolist())

# '부산'이 들어있는 문자열 컬럼 탐색
busan_cols = []

for col in boundary_all.columns:
    if col == "geometry":
        continue

    s = boundary_all[col].astype("string")
    count = s.str.contains("부산", na=False).sum()

    if count > 0:
        busan_cols.append((col, count))

        print(f"\n[{col}] 부산 포함 행 수: {count}")
        print(s[s.str.contains("부산", na=False)].drop_duplicates().head(10).tolist())

print("\n부산 식별 가능 컬럼:", busan_cols)

In [ ]:
# 부산광역시 행정동 경계 추출

if len(busan_cols) == 0:
    raise ValueError("'부산'을 포함하는 컬럼을 찾지 못했습니다.")

filter_col = max(
    busan_cols,
    key=lambda x: x[1]
)[0]

busan_boundary = boundary_all[
    boundary_all[filter_col]
    .astype("string")
    .str.contains("부산", na=False)
].copy()

print("사용한 필터 컬럼:", filter_col)
print("부산 경계 행 수:", len(busan_boundary))
print("CRS:", busan_boundary.crs)

display(busan_boundary.head())

In [ ]:
# 부산 행정동 경계 저장
boundary_path = (
    BASE_PATH
    + "02_processed_data/부산_행정동경계_20251231.geojson"
)

busan_boundary.to_file(
    boundary_path,
    driver="GeoJSON"
)

print("저장 완료:", boundary_path)
print("저장 행 수:", len(busan_boundary))

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

# 지도 표시용 후보 생성
map_candidates = final.dropna(
    subset=["x", "y"]
).copy()

candidate_gdf = gpd.GeoDataFrame(
    map_candidates,
    geometry=gpd.points_from_xy(
        map_candidates["x"],
        map_candidates["y"]
    ),
    crs="EPSG:4326"
)

print("전체 최종 후보:", len(final))
print("지도 표시 후보:", len(candidate_gdf))
print("좌표 미확보:", len(final) - len(candidate_gdf))

print("\n[지도 표시 유형별 건수]")
print(candidate_gdf["대표추천유형"].value_counts())

In [ ]:
# 활용유형별 최종 우선 검토 대상 분포 지도
fig, ax = plt.subplots(figsize=(9, 10))

# 부산 행정동 경계
busan_boundary.plot(
    ax=ax,
    facecolor="#F7F7F7",
    edgecolor="#CFCFCF",
    linewidth=0.45
)

type_styles = {
    "아동·가족 생활지원형": {
        "color": "#F4A261",
        "marker": "o"
    },
    "고령 생활지원형": {
        "color": "#457B9D",
        "marker": "^"
    },
    "활동·교류 지원형": {
        "color": "#2A9D8F",
        "marker": "s"
    }
}

for type_name, style in type_styles.items():

    temp = candidate_gdf[
        candidate_gdf["대표추천유형"] == type_name
    ]

    temp.plot(
        ax=ax,
        color=style["color"],
        marker=style["marker"],
        markersize=30,
        alpha=0.8,
        label=f"{type_name} ({len(temp)}건)"
    )

ax.set_title(
    "부산 유휴 공유재산 1차 우선 검토 대상",
    fontsize=16,
    pad=15
)

ax.legend(
    loc="lower left",
    frameon=False,
    fontsize=9
)

ax.set_axis_off()

plt.tight_layout()

output_path = (
    BASE_PATH
    + "04_visualization/최종후보_분포지도_초안.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("저장 완료:", output_path)

In [ ]:
# 지도에서 제외된 좌표 미확보 후보 확인
missing_coord = final[
    final[["x", "y"]].isna().any(axis=1)
][
    [
        "대표추천유형",
        "gugun",
        "행정동명",
        "lctnAddr",
        "zoneArea"
    ]
]

display(missing_coord)

In [ ]:
# Colab 한글 폰트 설정
!apt-get -qq install fonts-nanum

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
font_name = fm.FontProperties(fname=font_path).get_name()

plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

print("사용 폰트:", font_name)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()

mpl.rcParams["font.family"] = font_name
mpl.rcParams["axes.unicode_minus"] = False

print("사용 폰트:", font_name)
print("폰트 존재 여부:", os.path.exists(font_path))

In [ ]:
from matplotlib.font_manager import FontProperties

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
font_prop = FontProperties(fname=font_path)
font_prop_bold = FontProperties(fname=font_path, weight="bold")

fig, ax = plt.subplots(figsize=(9, 10))

busan_boundary.plot(
    ax=ax,
    facecolor="#F8F8F8",
    edgecolor="#D0D0D0",
    linewidth=0.45
)

type_styles = {
    "아동·가족 생활지원형": {
        "color": "#F4A261",
        "marker": "o"
    },
    "고령 생활지원형": {
        "color": "#457B9D",
        "marker": "^"
    },
    "활동·교류 지원형": {
        "color": "#2A9D8F",
        "marker": "s"
    }
}

for type_name, style in type_styles.items():

    temp = candidate_gdf[
        candidate_gdf["대표추천유형"] == type_name
    ]

    temp.plot(
        ax=ax,
        color=style["color"],
        marker=style["marker"],
        markersize=32,
        alpha=0.82,
        label=f"{type_name} ({len(temp)}건)"
    )

ax.set_title(
    "지역수요와 재산조건이 함께 높은 우선 검토 대상",
    fontproperties=font_prop_bold,
    fontsize=16,
    pad=15
)

ax.text(
    0.01,
    0.97,
    "최종 186건 중 좌표 확인 184건 표시",
    transform=ax.transAxes,
    fontproperties=font_prop,
    fontsize=9,
    va="top"
)

ax.legend(
    loc="lower left",
    frameon=False,
    prop=font_prop
)

ax.set_axis_off()

plt.tight_layout()

output_path = (
    BASE_PATH
    + "04_visualization/최종후보_분포지도.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("저장 완료:", output_path)

In [ ]:
save_path = BASE_PATH + "03_analysis_output/최종우선검토대상_186.csv"

candidate_gdf.drop(columns="geometry", errors="ignore").to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", save_path)

In [ ]:
# 최종 186건 전체 기준 요약표
summary_type = (
    final["대표추천유형"]
    .value_counts()
    .rename_axis("대표추천유형")
    .reset_index(name="후보수")
)

summary_gugun = (
    final["gugun"]
    .value_counts()
    .rename_axis("gugun")
    .reset_index(name="후보수")
)

summary_dong = (
    final["행정동명"]
    .value_counts()
    .rename_axis("행정동명")
    .reset_index(name="후보수")
)

print("[유형별 후보 수 - 최종 186건]")
display(summary_type)

print("\n[구군별 후보 수 - 최종 186건]")
display(summary_gugun)

print("\n[행정동별 후보 수 - 최종 186건]")
display(summary_dong.head(10))

# 기존 파일 덮어쓰기
summary_type.to_csv(
    BASE_PATH + "03_analysis_output/요약_유형별후보수.csv",
    index=False,
    encoding="utf-8-sig"
)

summary_gugun.to_csv(
    BASE_PATH + "03_analysis_output/요약_구군별후보수.csv",
    index=False,
    encoding="utf-8-sig"
)

summary_dong.to_csv(
    BASE_PATH + "03_analysis_output/요약_행정동별후보수.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n186건 기준 요약표 저장 완료")

In [ ]:
summary_type.to_csv(BASE_PATH + "03_analysis_output/요약_유형별후보수.csv", index=False, encoding="utf-8-sig")
summary_gugun.to_csv(BASE_PATH + "03_analysis_output/요약_구군별후보수.csv", index=False, encoding="utf-8-sig")
summary_dong.to_csv(BASE_PATH + "03_analysis_output/요약_행정동별후보수.csv", index=False, encoding="utf-8-sig")

print("요약표 저장 완료")

In [ ]:
top_child = (
    final[final["대표추천유형"] == "아동·가족 생활지원형"]
    .sort_values(
        ["아동_우선검토균형도", "zoneArea"],
        ascending=[False, False]
    )
    [[
        "gugun", "행정동명", "lctnAddr", "zoneArea",
        "대표추천유형", "아동_우선검토균형도", "추천근거"
    ]]
    .head(10)
)

top_old = (
    final[final["대표추천유형"] == "고령 생활지원형"]
    .sort_values(
        ["고령_우선검토균형도", "zoneArea"],
        ascending=[False, False]
    )
    [[
        "gugun", "행정동명", "lctnAddr", "zoneArea",
        "대표추천유형", "고령_우선검토균형도", "추천근거"
    ]]
    .head(10)
)

top_exchange = (
    final[final["대표추천유형"] == "활동·교류 지원형"]
    .sort_values(
        ["활동교류_우선검토균형도", "zoneArea"],
        ascending=[False, False]
    )
    [[
        "gugun", "행정동명", "lctnAddr", "zoneArea",
        "대표추천유형", "활동교류_우선검토균형도", "추천근거"
    ]]
    .head(10)
)

top_child.to_csv(
    BASE_PATH + "03_analysis_output/상위후보_아동가족.csv",
    index=False,
    encoding="utf-8-sig"
)

top_old.to_csv(
    BASE_PATH + "03_analysis_output/상위후보_고령.csv",
    index=False,
    encoding="utf-8-sig"
)

top_exchange.to_csv(
    BASE_PATH + "03_analysis_output/상위후보_활동교류.csv",
    index=False,
    encoding="utf-8-sig"
)

print("186건 기준 유형별 상위 후보 저장 완료")

In [ ]:
top_child.to_csv(BASE_PATH + "03_analysis_output/상위후보_아동가족.csv", index=False, encoding="utf-8-sig")
top_old.to_csv(BASE_PATH + "03_analysis_output/상위후보_고령.csv", index=False, encoding="utf-8-sig")
top_exchange.to_csv(BASE_PATH + "03_analysis_output/상위후보_활동교류.csv", index=False, encoding="utf-8-sig")

print("유형별 상위 후보 저장 완료")

In [ ]:
from matplotlib.patches import FancyBboxPatch
from matplotlib.font_manager import FontProperties

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
font_prop = FontProperties(fname=font_path)
font_prop_bold = FontProperties(fname=font_path, weight="bold")

steps = [
    ("10,102건", "부산 유휴 공유재산"),
    ("10,042건", "행정동·지역수요 연결"),
    ("1,572건", "1개 이상 활용유형 후보"),
    ("186건", "1차 우선 검토 대상")
]

fig, ax = plt.subplots(figsize=(5, 7))

ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.axis("off")

ys = [10.0, 7.2, 4.4, 1.6]

for i, ((number, label), y) in enumerate(zip(steps, ys)):

    box = FancyBboxPatch(
        (1.5, y - 0.8),
        7,
        1.6,
        boxstyle="round,pad=0.15,rounding_size=0.2",
        facecolor="#F7F7F7" if i < 3 else "#E6F2EF",
        edgecolor="#CCCCCC" if i < 3 else "#2A9D8F",
        linewidth=1.3
    )

    ax.add_patch(box)

    ax.text(
        5,
        y + 0.2,
        number,
        ha="center",
        va="center",
        fontsize=19,
        fontproperties=font_prop_bold
    )

    ax.text(
        5,
        y - 0.35,
        label,
        ha="center",
        va="center",
        fontsize=10,
        fontproperties=font_prop
    )

    if i < len(steps) - 1:
        ax.annotate(
            "",
            xy=(5, ys[i + 1] + 0.95),
            xytext=(5, y - 0.95),
            arrowprops=dict(
                arrowstyle="->",
                linewidth=1.2,
                color="#888888"
            )
        )

ax.set_title(
    "10,102건에서 186건까지",
    fontsize=16,
    fontproperties=font_prop_bold,
    pad=15
)

plt.tight_layout()

output_path = (
    BASE_PATH
    + "04_visualization/분석흐름_10102_to_186.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("저장 완료:", output_path)

In [ ]:
from matplotlib.patches import FancyBboxPatch

type_info = [
    {
        "name": "아동·가족 생활지원형",
        "count": 73,
        "share": 73 / 186 * 100,
        "desc": "20대 미만 활동인구의\n규모와 상대적 구성비를 함께 고려",
        "color": "#F4A261"
    },
    {
        "name": "고령 생활지원형",
        "count": 96,
        "share": 96 / 186 * 100,
        "desc": "60대 이상 활동인구의\n규모와 상대적 구성비를 함께 고려",
        "color": "#457B9D"
    },
    {
        "name": "활동·교류 지원형",
        "count": 17,
        "share": 17 / 186 * 100,
        "desc": "전체 활동규모와\n직장·방문활동 특화도를 함께 고려",
        "color": "#2A9D8F"
    }
]

fig, ax = plt.subplots(figsize=(7, 5.2))

ax.set_xlim(0, 12)
ax.set_ylim(0, 9)
ax.axis("off")

ys = [6.6, 3.9, 1.2]

for info, y in zip(type_info, ys):

    box = FancyBboxPatch(
        (0.8, y),
        10.4,
        1.9,
        boxstyle="round,pad=0.15,rounding_size=0.18",
        facecolor="white",
        edgecolor=info["color"],
        linewidth=1.7
    )

    ax.add_patch(box)

    # 유형명
    ax.text(
        1.3,
        y + 1.35,
        info["name"],
        fontsize=13,
        fontproperties=font_prop_bold,
        color=info["color"],
        va="center"
    )

    # 건수
    ax.text(
        9.9,
        y + 1.28,
        f'{info["count"]}건',
        fontsize=18,
        fontproperties=font_prop_bold,
        ha="right",
        va="center"
    )

    # 비율
    ax.text(
        10.0,
        y + 0.72,
        f'{info["share"]:.1f}%',
        fontsize=10,
        fontproperties=font_prop,
        ha="right",
        va="center",
        color="#666666"
    )

    # 설명
    ax.text(
        1.3,
        y + 0.55,
        info["desc"],
        fontsize=9.5,
        fontproperties=font_prop,
        va="center",
        linespacing=1.4
    )

ax.set_title(
    "지역수요에 따른 3가지 활용방향",
    fontsize=16,
    fontproperties=font_prop_bold,
    pad=15
)

ax.text(
    6,
    0.15,
    "※ 186건 모두 '대' 지목이면서 전체 유휴재산 면적 상위 25%에 해당하는 1차 우선 검토군",
    ha="center",
    fontsize=8.5,
    color="#666666",
    fontproperties=font_prop
)

plt.tight_layout()

output_path = (
    BASE_PATH
    + "04_visualization/활용유형_3개요약.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("저장 완료:", output_path)

In [ ]:
# 유형별 서로 다른 행정동에서 대표 재산 2건씩 선정

score_cols = {
    "아동·가족 생활지원형": "아동_우선검토균형도",
    "고령 생활지원형": "고령_우선검토균형도",
    "활동·교류 지원형": "활동교류_우선검토균형도"
}

representative_list = []

for type_name, score_col in score_cols.items():

    temp = (
        final[
            final["대표추천유형"] == type_name
        ]
        .sort_values(
            [score_col, "zoneArea"],
            ascending=[False, False]
        )
        .drop_duplicates(
            subset=["행정동명"],
            keep="first"
        )
        .head(2)
        .copy()
    )

    temp["대표유형"] = type_name
    temp["대표점수"] = temp[score_col]

    representative_list.append(temp)

representative_6 = pd.concat(
    representative_list,
    ignore_index=True
)

display(
    representative_6[
        [
            "대표유형",
            "gugun",
            "행정동명",
            "lctnAddr",
            "zoneArea",
            "대표점수"
        ]
    ]
)

In [ ]:
from matplotlib.patches import FancyBboxPatch
from matplotlib.font_manager import FontProperties

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
font_prop = FontProperties(fname=font_path)
font_prop_bold = FontProperties(fname=font_path, weight="bold")

type_colors = {
    "아동·가족 생활지원형": "#F4A261",
    "고령 생활지원형": "#457B9D",
    "활동·교류 지원형": "#2A9D8F"
}

fig, ax = plt.subplots(figsize=(10, 7))

ax.set_xlim(0, 14)
ax.set_ylim(0, 11)
ax.axis("off")

type_order = [
    "아동·가족 생활지원형",
    "고령 생활지원형",
    "활동·교류 지원형"
]

row_y = {
    "아동·가족 생활지원형": 7.3,
    "고령 생활지원형": 4.2,
    "활동·교류 지원형": 1.1
}

for type_name in type_order:

    temp = representative_6[
        representative_6["대표유형"] == type_name
    ].reset_index(drop=True)

    color = type_colors[type_name]
    y = row_y[type_name]

    # 유형명
    ax.text(
        0.5,
        y + 2.2,
        type_name,
        fontsize=13,
        color=color,
        fontproperties=font_prop_bold
    )

    for i, row in temp.iterrows():

        x = 0.5 + i * 6.7

        box = FancyBboxPatch(
            (x, y),
            6.1,
            1.85,
            boxstyle="round,pad=0.12,rounding_size=0.15",
            facecolor="white",
            edgecolor=color,
            linewidth=1.4
        )

        ax.add_patch(box)

        # 행정동
        ax.text(
            x + 0.3,
            y + 1.45,
            row["행정동명"],
            fontsize=11,
            color=color,
            fontproperties=font_prop_bold
        )

        # 주소는 부산광역시 제거
        short_addr = str(row["lctnAddr"]).replace(
            "부산광역시 ", ""
        )

        ax.text(
            x + 0.3,
            y + 0.92,
            short_addr,
            fontsize=8.5,
            fontproperties=font_prop
        )

        ax.text(
            x + 0.3,
            y + 0.43,
            f'{row["zoneArea"]:,.1f}㎡ · 대',
            fontsize=9.5,
            fontproperties=font_prop_bold
        )

        ax.text(
            x + 5.75,
            y + 0.43,
            f'수요 {row["대표점수"]:.3f}',
            fontsize=8.5,
            color="#666666",
            ha="right",
            fontproperties=font_prop
        )

ax.set_title(
    "활용유형별 대표 우선 검토 재산",
    fontsize=16,
    fontproperties=font_prop_bold,
    pad=15
)

ax.text(
    7,
    0.25,
    "※ 대표 사례는 결과 해석을 위한 예시이며, 최종 1차 우선 검토 대상은 총 186건",
    ha="center",
    fontsize=8.5,
    color="#666666",
    fontproperties=font_prop
)

plt.tight_layout()

output_path = (
    BASE_PATH
    + "04_visualization/활용유형별_대표재산.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("저장 완료:", output_path)

In [ ]:
import os

print("[03_analysis_output 파일]")
for file in os.listdir(
    BASE_PATH + "03_analysis_output/"
):
    print(file)

In [ ]:
import pandas as pd
import numpy as np

# 전체 유휴재산 + 지역수요 결합 데이터
idle_all = pd.read_csv(
    BASE_PATH + "02_processed_data/유휴공유재산_지역수요결합.csv",
    encoding="utf-8-sig"
)

profile_all = pd.read_csv(
    BASE_PATH + "02_processed_data/지역수요프로필_행정동별.csv",
    encoding="utf-8-sig"
)

pareto = pd.read_csv(
    BASE_PATH + "03_analysis_output/활용유형별_Pareto_후보행정동.csv",
    encoding="utf-8-sig"
)

# 행정동코드 형식 통일
def normalize_code(series):
    return (
        pd.to_numeric(series, errors="coerce")
        .astype("Int64")
        .astype("string")
    )

idle_all["행정동코드"] = normalize_code(idle_all["행정동코드"])
profile_all["행정동코드"] = normalize_code(profile_all["행정동코드"])
pareto["행정동코드"] = normalize_code(pareto["행정동코드"])

print("전체 재산:", len(idle_all))
print("지역수요 프로필:", len(profile_all))
print("Pareto 행:", len(pareto))

print("\n[Pareto 유형별]")
print(pareto["활용유형"].value_counts())

In [ ]:
# 시각화용 재산조건 재구성
idle_all["land_type_clean"] = (
    idle_all["land_type"]
    .astype("string")
    .str.replace(r"^\d+-", "", regex=True)
)

idle_all["area_valid"] = idle_all["zoneArea"] > 0

area_q75 = idle_all.loc[
    idle_all["area_valid"],
    "zoneArea"
].quantile(0.75)

idle_all["비도로_상위25pct"] = (
    idle_all["area_valid"]
    & idle_all["land_type_clean"].notna()
    & (idle_all["land_type_clean"] != "도로")
    & (idle_all["zoneArea"] >= area_q75)
)

dong_property_viz = (
    idle_all.dropna(subset=["행정동코드"])
    .groupby(["행정동코드", "행정동명"])
    .agg(
        유휴재산수=("lctnAddr", "size"),
        비도로_상위25pct_필지수=("비도로_상위25pct", "sum")
    )
    .reset_index()
)

print("면적 상위25% 기준:", area_q75, "㎡")
print("유휴재산 존재 행정동:", len(dong_property_viz))

In [ ]:
# 부산 전체 기준 구성비
city_under20_share = (
    profile_all["20대미만_활동인구"].sum()
    / profile_all["연령합계_전체활동인구"].sum()
)

city_over60_share = (
    profile_all["60대이상_활동인구"].sum()
    / profile_all["연령합계_전체활동인구"].sum()
)

city_work_share = (
    profile_all["평균직장인구수"].sum()
    / profile_all["전체활동인구"].sum()
)

city_visit_share = (
    profile_all["평균방문인구수"].sum()
    / profile_all["전체활동인구"].sum()
)

# 상대특성
profile_all["20대미만_LQ"] = (
    profile_all["20대미만_구성비"] / city_under20_share
)

profile_all["60대이상_LQ"] = (
    profile_all["60대이상_구성비"] / city_over60_share
)

profile_all["직장_LQ"] = (
    profile_all["직장인구_구성비"] / city_work_share
)

profile_all["방문_LQ"] = (
    profile_all["방문인구_구성비"] / city_visit_share
)

profile_all["활동교류_LQ"] = profile_all[
    ["직장_LQ", "방문_LQ"]
].max(axis=1)

# 206개 행정동 내 상대순위
profile_all["아동_규모_pct"] = (
    profile_all["20대미만_활동인구"].rank(pct=True)
)
profile_all["아동_구조_pct"] = (
    profile_all["20대미만_LQ"].rank(pct=True)
)

profile_all["고령_규모_pct"] = (
    profile_all["60대이상_활동인구"].rank(pct=True)
)
profile_all["고령_구조_pct"] = (
    profile_all["60대이상_LQ"].rank(pct=True)
)

profile_all["활동규모_pct"] = (
    profile_all["전체활동인구"].rank(pct=True)
)
profile_all["활동교류_특화_pct"] = (
    profile_all["활동교류_LQ"].rank(pct=True)
)

# 두 조건 중 낮은 값 = 수요균형도
profile_all["아동_수요균형도"] = profile_all[
    ["아동_규모_pct", "아동_구조_pct"]
].min(axis=1)

profile_all["고령_수요균형도"] = profile_all[
    ["고령_규모_pct", "고령_구조_pct"]
].min(axis=1)

profile_all["활동교류_수요균형도"] = profile_all[
    ["활동규모_pct", "활동교류_특화_pct"]
].min(axis=1)

dong_viz = dong_property_viz.merge(
    profile_all[
        [
            "행정동코드",
            "아동_수요균형도",
            "고령_수요균형도",
            "활동교류_수요균형도"
        ]
    ],
    on="행정동코드",
    how="left",
    validate="one_to_one"
)

print("시각화 대상 행정동:", len(dong_viz))
print("수요 결측:", dong_viz[
    [
        "아동_수요균형도",
        "고령_수요균형도",
        "활동교류_수요균형도"
    ]
].isna().sum().sum())

In [ ]:
from matplotlib.font_manager import FontProperties

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
font_prop = FontProperties(fname=font_path)
font_prop_bold = FontProperties(fname=font_path, weight="bold")

plot_settings = [
    (
        "아동·가족 생활지원형",
        "아동_수요균형도",
        "#F4A261",
        "Pareto_아동가족.png"
    ),
    (
        "고령 생활지원형",
        "고령_수요균형도",
        "#457B9D",
        "Pareto_고령.png"
    ),
    (
        "활동·교류 지원형",
        "활동교류_수요균형도",
        "#2A9D8F",
        "Pareto_활동교류.png"
    )
]

for type_name, demand_col, color, filename in plot_settings:

    fig, ax = plt.subplots(figsize=(7, 5))

    # 전체 172개 행정동
    ax.scatter(
        dong_viz[demand_col],
        dong_viz["비도로_상위25pct_필지수"],
        s=28,
        color="#D4D4D4",
        alpha=0.7,
        label="유휴재산 보유 행정동"
    )

    # Pareto 후보
    target_codes = pareto.loc[
        pareto["활용유형"] == type_name,
        "행정동코드"
    ]

    selected = dong_viz[
        dong_viz["행정동코드"].isin(target_codes)
    ].copy()

    ax.scatter(
        selected[demand_col],
        selected["비도로_상위25pct_필지수"],
        s=90,
        color=color,
        edgecolor="white",
        linewidth=1.0,
        zorder=3,
        label="Pareto 후보"
    )

    # Pareto 후보 이름 표시
    for _, row in selected.iterrows():
        ax.annotate(
            row["행정동명"].split()[-1],
            (
                row[demand_col],
                row["비도로_상위25pct_필지수"]
            ),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8.5,
            fontproperties=font_prop,
            color="#333333"
        )

    # Pareto 경계 연결
    selected_line = selected.sort_values(demand_col)

    ax.plot(
        selected_line[demand_col],
        selected_line["비도로_상위25pct_필지수"],
        linestyle="--",
        linewidth=1.1,
        color=color,
        alpha=0.7
    )

    ax.set_title(
        type_name,
        fontsize=14,
        fontproperties=font_prop_bold,
        pad=12
    )

    ax.set_xlabel(
        "지역수요 균형도 →",
        fontproperties=font_prop,
        fontsize=10
    )

    ax.set_ylabel(
        "상대적으로 큰 비도로 유휴재산 수 →",
        fontproperties=font_prop,
        fontsize=10
    )

    ax.grid(
        alpha=0.15,
        linewidth=0.7
    )

    ax.legend(
        frameon=False,
        prop=font_prop,
        fontsize=8
    )

    plt.tight_layout()

    output_path = (
        BASE_PATH
        + "04_visualization/"
        + filename
    )

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()

    print("저장 완료:", output_path)

In [ ]:
# 유휴 공유재산 자체의 구조 요약

area_bins = [
    "10㎡ 이하",
    "10~30㎡",
    "30~50㎡",
    "50~100㎡",
    "100~300㎡",
    "300~1000㎡",
    "1000㎡ 초과"
]

area_counts = [
    3180,
    2175,
    1330,
    1439,
    1170,
    539,
    269
]

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.barh(
    area_bins,
    area_counts,
    color="#B7C9D6"
)

ax.invert_yaxis()

for bar, value in zip(bars, area_counts):
    ax.text(
        value + 40,
        bar.get_y() + bar.get_height()/2,
        f"{value:,}건",
        va="center",
        fontsize=9,
        fontproperties=font_prop
    )

ax.axvline(
    0,
    color="#999999",
    linewidth=0.7
)

ax.set_title(
    "유휴 공유재산은 크기부터 서로 다르다",
    fontsize=15,
    fontproperties=font_prop_bold,
    pad=12
)

ax.text(
    0.99,
    0.05,
    "중앙값 27㎡  ·  상위 25% 기준 75㎡",
    transform=ax.transAxes,
    ha="right",
    fontsize=9.5,
    fontproperties=font_prop_bold,
    color="#555555"
)

ax.set_xlabel(
    "유휴재산 수",
    fontsize=9,
    fontproperties=font_prop
)

ax.tick_params(axis="both", labelsize=9)

for label in ax.get_yticklabels():
    label.set_fontproperties(font_prop)

for label in ax.get_xticklabels():
    label.set_fontproperties(font_prop)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

ax.grid(
    axis="x",
    alpha=0.15
)

plt.tight_layout()

output_path = (
    BASE_PATH
    + "04_visualization/유휴재산_면적구조.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("저장 완료:", output_path)

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.font_manager import FontProperties
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import os

# --------------------------------------------------
# 1. 파일 불러오기
# --------------------------------------------------

final_path = (
    BASE_PATH
    + "03_analysis_output/"
    + "최종_유휴공유재산_우선검토대상_186건.csv"
)

boundary_path = (
    BASE_PATH
    + "02_processed_data/"
    + "부산_행정동경계_20251231.geojson"
)

final_map = pd.read_csv(
    final_path,
    encoding="utf-8-sig"
)

busan_dong = gpd.read_file(
    boundary_path
)

print("최종 후보:", len(final_map))
print("경계 행정동:", len(busan_dong))

print("\n[최종 후보 컬럼]")
print(final_map.columns.tolist())

In [ ]:
# --------------------------------------------------
# 2. 좌표 컬럼 확인 및 GeoDataFrame 생성
# --------------------------------------------------

coordinate_candidates = [
    ("x", "y"),
    ("X", "Y"),
    ("longitude", "latitude"),
    ("lon", "lat"),
    ("경도", "위도")
]

x_col = None
y_col = None

for x_candidate, y_candidate in coordinate_candidates:
    if (
        x_candidate in final_map.columns
        and y_candidate in final_map.columns
    ):
        x_col = x_candidate
        y_col = y_candidate
        break

if x_col is None:
    raise ValueError(
        "좌표 컬럼을 찾지 못했습니다. "
        "위에서 출력된 컬럼명을 보여주세요."
    )

final_map[x_col] = pd.to_numeric(
    final_map[x_col],
    errors="coerce"
)

final_map[y_col] = pd.to_numeric(
    final_map[y_col],
    errors="coerce"
)

map_df = final_map.dropna(
    subset=[x_col, y_col]
).copy()

candidate_gdf = gpd.GeoDataFrame(
    map_df,
    geometry=gpd.points_from_xy(
        map_df[x_col],
        map_df[y_col]
    ),
    crs="EPSG:4326"
)

busan_dong = busan_dong.to_crs("EPSG:4326")

print("전체 최종 후보:", len(final_map))
print("지도 표시 가능:", len(candidate_gdf))

print("\n[지도 표시 유형별]")
print(
    candidate_gdf[
        "대표추천유형"
    ].value_counts()
)

In [ ]:
# 나눔고딕 설치 및 matplotlib 등록
!apt-get update -qq
!apt-get install -y fonts-nanum > /dev/null

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

print("폰트 존재:", os.path.exists(font_path))

fm.fontManager.addfont(font_path)

font_prop = fm.FontProperties(
    fname=font_path
)

font_bold = fm.FontProperties(
    fname=font_path,
    weight="bold"
)

plt.rcParams["font.family"] = font_prop.get_name()
plt.rcParams["axes.unicode_minus"] = False

print("사용 폰트:", font_prop.get_name())

In [ ]:
# --------------------------------------------------
# 3. 확대형 메인 지도 + 부산 전체 인셋맵
# --------------------------------------------------

font_path = (
    "/usr/share/fonts/truetype/nanum/"
    "NanumGothic.ttf"
)

font_prop = FontProperties(
    fname=font_path
)

font_bold = FontProperties(
    fname=font_path,
    weight="bold"
)

# 구·군 경계 생성
busan_gu = (
    busan_dong[
        ["sggnm", "geometry"]
    ]
    .dissolve(
        by="sggnm"
    )
    .reset_index()
)

# 후보지역 중심 자동 확대 범위
minx, miny, maxx, maxy = (
    candidate_gdf.total_bounds
)

x_range = maxx - minx
y_range = maxy - miny

x_padding = x_range * 0.08
y_padding = y_range * 0.10

zoom_minx = minx - x_padding
zoom_maxx = maxx + x_padding
zoom_miny = miny - y_padding
zoom_maxy = maxy + y_padding

# 유형별 시각 설정
style_dict = {
    "아동·가족 생활지원형": {
        "color": "#F4A261",
        "marker": "o",
        "label": "아동·가족"
    },
    "고령 생활지원형": {
        "color": "#457B9D",
        "marker": "^",
        "label": "고령"
    },
    "활동·교류 지원형": {
        "color": "#2A9D8F",
        "marker": "s",
        "label": "활동·교류"
    }
}

fig, ax = plt.subplots(
    figsize=(9, 7.5)
)

# 행정동 경계
busan_dong.plot(
    ax=ax,
    facecolor="#F7F8F9",
    edgecolor="#D7DDE1",
    linewidth=0.35,
    zorder=1
)

# 구·군 경계
busan_gu.plot(
    ax=ax,
    facecolor="none",
    edgecolor="#A7B0B7",
    linewidth=0.75,
    zorder=2
)

# 후보점
for type_name, style in style_dict.items():

    subset = candidate_gdf[
        candidate_gdf[
            "대표추천유형"
        ] == type_name
    ]

    ax.scatter(
        subset.geometry.x,
        subset.geometry.y,
        s=48,
        marker=style["marker"],
        color=style["color"],
        edgecolor="white",
        linewidth=0.55,
        alpha=0.88,
        zorder=5
    )

# 후보지역 중심 확대
ax.set_xlim(
    zoom_minx,
    zoom_maxx
)

ax.set_ylim(
    zoom_miny,
    zoom_maxy
)

ax.set_aspect(
    "equal",
    adjustable="box"
)

ax.axis("off")

# --------------------------------------------------
# 4. 부산 전체 위치 인셋맵
# --------------------------------------------------

ax_inset = inset_axes(
    ax,
    width="26%",
    height="26%",
    loc="upper left",
    borderpad=1.2
)

busan_dong.plot(
    ax=ax_inset,
    facecolor="#F5F6F7",
    edgecolor="#C7CDD2",
    linewidth=0.22
)

busan_gu.plot(
    ax=ax_inset,
    facecolor="none",
    edgecolor="#AAB2B8",
    linewidth=0.4
)

# 확대영역 표시
zoom_rect = Rectangle(
    (
        zoom_minx,
        zoom_miny
    ),
    zoom_maxx - zoom_minx,
    zoom_maxy - zoom_miny,
    fill=False,
    edgecolor="#4B5563",
    linewidth=1.0
)

ax_inset.add_patch(
    zoom_rect
)

ax_inset.set_aspect(
    "equal",
    adjustable="box"
)

ax_inset.axis("off")

# 인셋 테두리
for spine in ax_inset.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor("#D0D5D9")
    spine.set_linewidth(0.7)

# --------------------------------------------------
# 5. 범례
# --------------------------------------------------

legend_handles = []

for type_name, style in style_dict.items():

    count = len(
        candidate_gdf[
            candidate_gdf[
                "대표추천유형"
            ] == type_name
        ]
    )

    legend_handles.append(
        Line2D(
            [0],
            [0],
            marker=style["marker"],
            linestyle="None",
            markerfacecolor=style["color"],
            markeredgecolor="white",
            markeredgewidth=0.7,
            markersize=8,
            label=f'{style["label"]} {count}건'
        )
    )

legend = ax.legend(
    handles=legend_handles,
    loc="lower left",
    frameon=False,
    prop=font_prop,
    fontsize=9,
    ncol=3,
    handletextpad=0.4,
    columnspacing=1.2,
    borderaxespad=0.6
)

# --------------------------------------------------
# 6. 저장
# --------------------------------------------------

# 인셋맵 때문에 tight_layout 대신 여백 직접 조정
plt.subplots_adjust(
    left=0.02,
    right=0.98,
    top=0.98,
    bottom=0.04
)

output_with_legend = (
    BASE_PATH
    + "04_visualization/"
    + "최종후보_분포지도_확대형.png"
)

plt.savefig(
    output_with_legend,
    dpi=350,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print(
    "범례 포함 지도 저장 완료:",
    output_with_legend
)

In [ ]:
# --------------------------------------------------
# 7. 범례 없는 버전
# --------------------------------------------------

fig, ax = plt.subplots(
    figsize=(9, 7.5)
)

busan_dong.plot(
    ax=ax,
    facecolor="#F7F8F9",
    edgecolor="#D7DDE1",
    linewidth=0.35,
    zorder=1
)

busan_gu.plot(
    ax=ax,
    facecolor="none",
    edgecolor="#A7B0B7",
    linewidth=0.75,
    zorder=2
)

for type_name, style in style_dict.items():

    subset = candidate_gdf[
        candidate_gdf[
            "대표추천유형"
        ] == type_name
    ]

    ax.scatter(
        subset.geometry.x,
        subset.geometry.y,
        s=52,
        marker=style["marker"],
        color=style["color"],
        edgecolor="white",
        linewidth=0.55,
        alpha=0.9,
        zorder=5
    )

ax.set_xlim(
    zoom_minx,
    zoom_maxx
)

ax.set_ylim(
    zoom_miny,
    zoom_maxy
)

ax.set_aspect(
    "equal",
    adjustable="box"
)

ax.axis("off")

# 부산 전체 인셋맵
ax_inset = inset_axes(
    ax,
    width="26%",
    height="26%",
    loc="upper left",
    borderpad=1.2
)

busan_dong.plot(
    ax=ax_inset,
    facecolor="#F5F6F7",
    edgecolor="#C7CDD2",
    linewidth=0.22
)

busan_gu.plot(
    ax=ax_inset,
    facecolor="none",
    edgecolor="#AAB2B8",
    linewidth=0.4
)

zoom_rect = Rectangle(
    (
        zoom_minx,
        zoom_miny
    ),
    zoom_maxx - zoom_minx,
    zoom_maxy - zoom_miny,
    fill=False,
    edgecolor="#4B5563",
    linewidth=1.0
)

ax_inset.add_patch(
    zoom_rect
)

ax_inset.set_aspect(
    "equal",
    adjustable="box"
)

ax_inset.axis("off")

# --------------------------------------------------
# 범례 없는 지도 저장
# --------------------------------------------------

plt.subplots_adjust(
    left=0.02,
    right=0.98,
    top=0.98,
    bottom=0.04
)

output_no_legend = (
    BASE_PATH
    + "04_visualization/"
    + "최종후보_분포지도_확대형_범례없음.png"
)

plt.savefig(
    output_no_legend,
    dpi=350,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print(
    "범례 없는 지도 저장 완료:",
    output_no_legend
)

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# --------------------------------------------------
# 1. 데이터 불러오기
# --------------------------------------------------

final_path = (
    BASE_PATH
    + "03_analysis_output/"
    + "최종_유휴공유재산_우선검토대상_186건.csv"
)

boundary_path = (
    BASE_PATH
    + "02_processed_data/"
    + "부산_행정동경계_20251231.geojson"
)

final_map = pd.read_csv(
    final_path,
    encoding="utf-8-sig"
)

busan_dong = gpd.read_file(
    boundary_path
).to_crs("EPSG:4326")


# --------------------------------------------------
# 2. 좌표 컬럼 자동 확인
# --------------------------------------------------

coordinate_candidates = [
    ("x", "y"),
    ("X", "Y"),
    ("longitude", "latitude"),
    ("lon", "lat"),
    ("경도", "위도")
]

x_col = None
y_col = None

for x_candidate, y_candidate in coordinate_candidates:
    if (
        x_candidate in final_map.columns
        and y_candidate in final_map.columns
    ):
        x_col = x_candidate
        y_col = y_candidate
        break

if x_col is None:
    raise ValueError(
        "좌표 컬럼을 찾지 못했습니다."
    )

final_map[x_col] = pd.to_numeric(
    final_map[x_col],
    errors="coerce"
)

final_map[y_col] = pd.to_numeric(
    final_map[y_col],
    errors="coerce"
)

map_df = final_map.dropna(
    subset=[x_col, y_col]
).copy()

candidate_gdf = gpd.GeoDataFrame(
    map_df,
    geometry=gpd.points_from_xy(
        map_df[x_col],
        map_df[y_col]
    ),
    crs="EPSG:4326"
)


# --------------------------------------------------
# 3. 지도 표시 건수 확인
# --------------------------------------------------

print("전체 최종 후보:", len(final_map))
print("좌표 확인 후보:", len(candidate_gdf))

print("\n[좌표 확인 후보 - 구군별]")
print(
    candidate_gdf["gugun"]
    .value_counts()
)

print("\n[좌표 확인 후보 - 활용유형별]")
print(
    candidate_gdf["대표추천유형"]
    .value_counts()
)


# --------------------------------------------------
# 4. 구군 경계 생성
# --------------------------------------------------

busan_gu = (
    busan_dong[
        ["sggnm", "geometry"]
    ]
    .dissolve(
        by="sggnm"
    )
    .reset_index()
)


# --------------------------------------------------
# 5. 활용유형 스타일
# --------------------------------------------------

style_dict = {
    "아동·가족 생활지원형": {
        "color": "#F4A261",
        "marker": "o"
    },
    "고령 생활지원형": {
        "color": "#457B9D",
        "marker": "^"
    },
    "활동·교류 지원형": {
        "color": "#2A9D8F",
        "marker": "s"
    }
}


# --------------------------------------------------
# 6. 메인지도 범위
#    부산진구 + 영도구 중심
# --------------------------------------------------

main_gus = [
    "부산진구",
    "영도구"
]

main_boundary = busan_gu[
    busan_gu["sggnm"].isin(main_gus)
].copy()

minx, miny, maxx, maxy = (
    main_boundary.total_bounds
)

x_range = maxx - minx
y_range = maxy - miny

main_minx = minx - x_range * 0.10
main_maxx = maxx + x_range * 0.10
main_miny = miny - y_range * 0.07
main_maxy = maxy + y_range * 0.07


# --------------------------------------------------
# 7. 메인지도 생성
# --------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8.4, 8.0)
)

# 전체 행정동 배경
busan_dong.plot(
    ax=ax,
    facecolor="#F7F8F9",
    edgecolor="#DCE1E5",
    linewidth=0.32,
    zorder=1
)

# 구군 경계
busan_gu.plot(
    ax=ax,
    facecolor="none",
    edgecolor="#9CAAB3",
    linewidth=0.8,
    zorder=2
)

# 메인지도 후보
main_candidates = candidate_gdf[
    candidate_gdf["gugun"].isin(main_gus)
].copy()

for type_name, style in style_dict.items():

    subset = main_candidates[
        main_candidates[
            "대표추천유형"
        ] == type_name
    ]

    ax.scatter(
        subset.geometry.x,
        subset.geometry.y,
        s=70,
        marker=style["marker"],
        color=style["color"],
        edgecolor="white",
        linewidth=0.65,
        alpha=0.92,
        zorder=5
    )

ax.set_xlim(
    main_minx,
    main_maxx
)

ax.set_ylim(
    main_miny,
    main_maxy
)

ax.set_aspect(
    "equal",
    adjustable="box"
)

ax.axis("off")


# --------------------------------------------------
# 8. 해운대구 인셋맵
# --------------------------------------------------

haeundae_boundary = busan_gu[
    busan_gu["sggnm"] == "해운대구"
].copy()

h_minx, h_miny, h_maxx, h_maxy = (
    haeundae_boundary.total_bounds
)

h_xrange = h_maxx - h_minx
h_yrange = h_maxy - h_miny

ax_inset = inset_axes(
    ax,
    width="29%",
    height="29%",
    loc="upper right",
    borderpad=0.7
)

# 해운대 행정동만
haeundae_dong = busan_dong[
    busan_dong["sggnm"] == "해운대구"
]

haeundae_dong.plot(
    ax=ax_inset,
    facecolor="#F7F8F9",
    edgecolor="#D1D8DD",
    linewidth=0.35,
    zorder=1
)

haeundae_boundary.plot(
    ax=ax_inset,
    facecolor="none",
    edgecolor="#94A3AD",
    linewidth=0.8,
    zorder=2
)

haeundae_candidates = candidate_gdf[
    candidate_gdf["gugun"] == "해운대구"
].copy()

for type_name, style in style_dict.items():

    subset = haeundae_candidates[
        haeundae_candidates[
            "대표추천유형"
        ] == type_name
    ]

    ax_inset.scatter(
        subset.geometry.x,
        subset.geometry.y,
        s=56,
        marker=style["marker"],
        color=style["color"],
        edgecolor="white",
        linewidth=0.6,
        alpha=0.95,
        zorder=5
    )

ax_inset.set_xlim(
    h_minx - h_xrange * 0.08,
    h_maxx + h_xrange * 0.08
)

ax_inset.set_ylim(
    h_miny - h_yrange * 0.08,
    h_maxy + h_yrange * 0.08
)

ax_inset.set_aspect(
    "equal",
    adjustable="box"
)

ax_inset.set_xticks([])
ax_inset.set_yticks([])

# 인셋 테두리
for spine in ax_inset.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor("#B8C1C8")
    spine.set_linewidth(0.8)


# --------------------------------------------------
# 9. 여백 최소화 및 저장
# --------------------------------------------------

plt.subplots_adjust(
    left=0.01,
    right=0.99,
    top=0.99,
    bottom=0.01
)

output_path = (
    BASE_PATH
    + "04_visualization/"
    + "최종후보_분포지도_포스터용_확대.png"
)

plt.savefig(
    output_path,
    dpi=400,
    bbox_inches="tight",
    pad_inches=0.02,
    facecolor="white"
)

plt.show()

print(
    "\n저장 완료:",
    output_path
)